# 🎤 DSM-ASR: Mimi + Qwen3-0.6B

**Delayed Streams Modeling** for Arabic ASR using:
- **Mimi** audio tokenizer (12.5Hz, 8 codebooks)
- **Qwen3-0.6B-Base** transformer backbone

Inspired by [Kyutai STT](https://huggingface.co/kyutai/stt-1b-en_fr)

---
**Runtime**: A100 40GB recommended

## 1. Setup

In [ ]:
# Check GPU
!nvidia-smi
import torch
print(f"\n✅ CUDA: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

In [ ]:
# Clone your repo (CHANGE THIS to your GitHub repo URL)
REPO_URL = "https://github.com/YOUR_USERNAME/DSM.git"  # <-- CHANGE THIS

!git clone {REPO_URL} /content/DSM
%cd /content/DSM

In [ ]:
# Install dependencies
!pip install -q torch transformers>=4.51.0 datasets[audio] accelerate \
    soundfile librosa whisper-timestamped tqdm jiwer wandb

In [ ]:
# Login to HuggingFace (for your private dataset)
from huggingface_hub import login

# Option 1: Enter token interactively
login()

# Option 2: Use your token directly (uncomment and paste your token)
# login(token="hf_YOUR_TOKEN_HERE")

## 2. Preprocess Dataset

This step:
1. Loads your dataset from HuggingFace
2. Generates word-level timestamps using whisper-timestamped
3. Pre-encodes all audio with Mimi (saves codebook indices)

⏱️ **First run takes time** (whisper + mimi on each sample). Results are saved to disk.

In [ ]:
# Test with a small number of samples first
!python data/prepare_timestamps.py --max_samples 20 --language ar

In [ ]:
# Check the preprocessed data
import json
with open("preprocessed_data/manifest.json") as f:
    manifest = json.load(f)
print(f"Processed: {manifest['total_processed']}")
print(f"Errors: {manifest['total_errors']}")
if manifest['samples']:
    s = manifest['samples'][0]
    print(f"\nFirst sample: {s['num_frames']} frames, {s['num_words']} words, {s['duration']:.1f}s")
    print(f"Text: {s['text'][:100]}...")

In [ ]:
# If the test looks good, preprocess the FULL dataset
# (uncomment when ready)
# !python data/prepare_timestamps.py --language ar

## 3. Quick Sanity Checks

In [ ]:
# Test: Dataset pipeline
!python data/dataset.py

In [ ]:
# Test: Model forward pass
!python model/dsm_asr.py

## 4. Train

**Two-phase training:**
1. **Phase 1** (epochs 1-2): Qwen3 backbone frozen, train audio embeddings only
2. **Phase 2** (epochs 3+): Full fine-tuning with lower LR

On A100 40GB with batch_size=4 and grad_accum=8, effective batch = 32.

In [ ]:
# Quick sanity check: 10 training steps
!python train.py --max_steps 10 --batch_size 2 --max_samples 20

In [ ]:
# Full training (adjust batch_size based on your GPU memory)
# A100 40GB -> batch_size=4 should work well
!python train.py \
    --batch_size 4 \
    --num_epochs 10 \
    --learning_rate 5e-5 \
    --output_dir ./output

In [ ]:
# Monitor training loss
import json
import matplotlib.pyplot as plt

try:
    with open("output/training_log.json") as f:
        log = json.load(f)
    steps = [e["step"] for e in log]
    losses = [e["loss"] for e in log]
    plt.figure(figsize=(10, 4))
    plt.plot(steps, losses)
    plt.xlabel("Step")
    plt.ylabel("Loss")
    plt.title("DSM-ASR Training Loss")
    plt.grid(True, alpha=0.3)
    plt.show()
except:
    print("No training log found yet")

## 5. Evaluate

In [ ]:
# Evaluate the best checkpoint
!python evaluate.py --checkpoint output/best --fast --output output/eval_results.json

## 6. Inference

In [ ]:
# Upload an audio file and transcribe it
from google.colab import files
uploaded = files.upload()
audio_file = list(uploaded.keys())[0]
print(f"Uploaded: {audio_file}")

In [ ]:
# Transcribe
!python inference.py --checkpoint output/best --audio {audio_file}

## 7. Save Model to HuggingFace (Optional)

In [ ]:
# Push best checkpoint to HuggingFace Hub
from huggingface_hub import HfApi

api = HfApi()
repo_id = "nadsoft/dsm-asr-arabic"  # Change to your repo

api.create_repo(repo_id, exist_ok=True)
api.upload_folder(
    folder_path="output/best",
    repo_id=repo_id,
    commit_message="DSM-ASR: Mimi + Qwen3-0.6B trained model",
)
print(f"✅ Uploaded to https://huggingface.co/{repo_id}")

## 8. Save to Google Drive (Backup)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!cp -r output/best /content/drive/MyDrive/dsm_asr_checkpoint/
print("✅ Saved to Google Drive")